# Fetching stock prices & PCA similarity

Use the `stonks` library to fetch the **last 2 years** of daily prices for the most-liquid US common stocks, pack them into a NumPy matrix, then explore **which stocks are similar in PCA space**.

Run from the repo root with the project env active so `stonks` is importable in editable mode.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stonks import get_prices, load_universe

%matplotlib inline


## Parameters

- `PERIOD`: history window — last 2 years.
- `TOP_N`: how many of the most-liquid stocks to keep.
- `FIELD`: which OHLCV field (`close` is split/dividend adjusted).

> The first call for a given `period`/`interval` downloads the entire common-stock universe and ranks it by traded dollar volume — expect a couple of minutes. It's cached to `data/`, so afterwards changing `TOP_N` or `FIELD` is instant.


In [ ]:
PERIOD = "2y"
INTERVAL = "1d"
TOP_N = 50          # bump to 1500 for the full set (free after the first run caches the universe)
FIELD = "close"


## Fetch prices


In [ ]:
prices = get_prices(top_n=TOP_N, period=PERIOD, interval=INTERVAL, field=FIELD)
prices


The result is an **N×T** DataFrame: rows are tickers (N), columns are dates (T). Some rows contain `NaN` (recently delisted / thinly traded names) — drop them for a clean matrix.


In [ ]:
prices_clean = prices.dropna()
X = prices_clean.to_numpy(dtype=float)
tickers = list(prices_clean.index)

print("X.shape:", X.shape, "| N stocks:", len(tickers))


In [ ]:
plt.plot(X.T)
plt.title("Raw close prices"); plt.xlabel("time"); plt.ylabel("price")


## Centre & covariance

Centre each stock's series by its own mean (remove the level, keep the shape), then build the stock×stock covariance, normalised by **T** (the time axis).


In [ ]:
X_centered = X - X.mean(axis=1, keepdims=True)
plt.plot(X_centered.T)
plt.title("Mean-centred prices"); plt.xlabel("time"); plt.ylabel("centred price")


In [ ]:
cov = (X_centered @ X_centered.T) / X_centered.shape[1]

plt.imshow(cov)
plt.colorbar(label="covariance")
plt.title("Stock × stock covariance"); plt.xlabel("stock"); plt.ylabel("stock")


## PCA via SVD

`cov` is symmetric positive-semidefinite, so decompose with **SVD** (equivalently `eigh`). Never `np.linalg.eig` — on a symmetric matrix it returns a complex dtype with tiny imaginary noise.


In [ ]:
U, S, Vt = np.linalg.svd(cov)
print("top singular values:", np.round(S[:5], 3))


## Stocks in PCA space

Because `cov` is N×N over **stocks**, each eigenvector `U[:,k]` is an N-vector with one entry per stock — so `(U[i,0], U[i,1])` is stock *i*'s coordinate on PC1/PC2. Each point below is a stock, annotated with its ticker.


In [ ]:
uni = load_universe()
NAMES = dict(zip(uni["ticker"], uni["name"]))

def label(t: str) -> str:
    return f"{t} ({NAMES.get(t, '')[:22]})"


In [ ]:
coords = U[:, :2]
fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(coords[:, 0], coords[:, 1], s=25)
for i, t in enumerate(tickers):
    ax.annotate(t, (coords[i, 0], coords[i, 1]), fontsize=7, alpha=0.8)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Stocks in PCA space (top 2 components)")


## Which stocks are similar?

Similarity = Euclidean distance in the top-`k` PCA space. Small distance ⇒ similar price-trajectory shape.


In [ ]:
k = 5  # number of PCs used to define similarity
Z = U[:, :k]
D = np.sqrt(((Z[:, None, :] - Z[None, :, :]) ** 2).sum(-1))
np.fill_diagonal(D, np.inf)  # ignore self

iu = np.triu_indices(len(tickers), k=1)
print("most similar pairs:")
for r in np.argsort(D[iu])[:10]:
    i, j = iu[0][r], iu[1][r]
    print(f"  d={D[i, j]:.3f}  {label(tickers[i]):30s} ~ {label(tickers[j])}")


In [ ]:
target = "AAPL"
i = tickers.index(target)
j = int(D[i].argmin())
print(f"{label(target)}'s nearest neighbour: {label(tickers[j])}  (d={D[i, j]:.3f})")


### Refinement: returns instead of prices

Above uses centred *prices*, so PC1 mostly captures overall trajectory shape. To cluster by co-movement (more sector-like), rebuild the covariance from log-returns:

```python
R = np.log(prices_clean / prices_clean.shift(axis=1))
X_centered = R.dropna(axis=1, how="all").fillna(0.0).to_numpy(dtype=float)
# then rerun the covariance / SVD / distance cells
```
